In [1]:
# ============================================================
# CIFAR-10 Continual Learning Mini-Project (Full Code Page)
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split, ConcatDataset
from torchvision import datasets, transforms

import numpy as np
import random

# -----------------------------
# 1. Setup
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if device.type == "cuda":
    torch.cuda.manual_seed_all(seed)

# -----------------------------
# 2. Data Loading & Transforms
# -----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

train_full = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_full  = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

print("Train size:", len(train_full))
print("Test size:", len(test_full))

# -----------------------------
# 3. Build Subset A (0–4) and B (5–9)
# -----------------------------
def get_indices_by_classes(dataset, class_list, max_per_class=None):
    targets = np.array(dataset.targets)
    indices = []
    for c in class_list:
        class_idx = np.where(targets == c)[0]
        if max_per_class is not None:
            class_idx = class_idx[:max_per_class]
        indices.extend(class_idx)
    return indices

classes_A = [0, 1, 2, 3, 4]
classes_B = [5, 6, 7, 8, 9]

max_per_class = 4000  # reduce if needed for speed

idx_A = get_indices_by_classes(train_full, classes_A, max_per_class)
idx_B = get_indices_by_classes(train_full, classes_B, max_per_class)

subset_A = Subset(train_full, idx_A)
subset_B = Subset(train_full, idx_B)

print("Subset A size:", len(subset_A))
print("Subset B size:", len(subset_B))

# Train/val split for A
val_ratio = 0.2
len_A = len(subset_A)
len_A_train = int((1 - val_ratio) * len_A)
len_A_val = len_A - len_A_train
subset_A_train, subset_A_val = random_split(subset_A, [len_A_train, len_A_val])

# Train/val split for B
len_B = len(subset_B)
len_B_train = int((1 - val_ratio) * len_B)
len_B_val = len_B - len_B_train
subset_B_train, subset_B_val = random_split(subset_B, [len_B_train, len_B_val])

print("A train/val:", len(subset_A_train), len(subset_A_val))
print("B train/val:", len(subset_B_train), len(subset_B_val))

# -----------------------------
# 4. DataLoaders
# -----------------------------
batch_size = 128

train_loader_A = DataLoader(subset_A_train, batch_size=batch_size, shuffle=True)
val_loader_A   = DataLoader(subset_A_val,   batch_size=batch_size, shuffle=False)

train_loader_B = DataLoader(subset_B_train, batch_size=batch_size, shuffle=True)
val_loader_B   = DataLoader(subset_B_val,   batch_size=batch_size, shuffle=False)

combined_train = ConcatDataset([subset_A_train, subset_B_train])
combined_val   = ConcatDataset([subset_A_val,   subset_B_val])

train_loader_combined = DataLoader(combined_train, batch_size=batch_size, shuffle=True)
val_loader_combined   = DataLoader(combined_val,   batch_size=batch_size, shuffle=False)

# -----------------------------
# 5. Simple CNN Model
# -----------------------------
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 8 * 8, 256)
        self.fc2   = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 32x16x16
        x = self.pool(F.relu(self.conv2(x)))  # 64x8x8
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def init_model():
    model = SimpleCNN(num_classes=10).to(device)
    return model

# -----------------------------
# 6. Train & Eval Helpers
# -----------------------------
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(targets).sum().item()
        total += targets.size(0)

    return running_loss / total, correct / total

def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * inputs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(targets).sum().item()
            total += targets.size(0)
    return running_loss / total, correct / total

criterion = nn.CrossEntropyLoss()

# -----------------------------
# 7. Initial Training on Subset A
# -----------------------------
model_A = init_model()
optimizer_A = optim.Adam(model_A.parameters(), lr=1e-3)

epochs_initial = 5  # adjust for time/compute

print("\n=== Initial Training on Subset A (classes 0–4) ===")
for epoch in range(epochs_initial):
    train_loss, train_acc = train_one_epoch(model_A, train_loader_A, optimizer_A, criterion)
    val_loss, val_acc = evaluate(model_A, val_loader_A, criterion)
    print(f"[Initial A] Epoch {epoch+1}/{epochs_initial} | "
          f"Train Acc: {train_acc:.3f} | Val Acc (A): {val_acc:.3f}")

# -----------------------------
# 8. Simulate Data Shift: Evaluate on Subset B
# -----------------------------
val_loss_B_init, val_acc_B_init = evaluate(model_A, val_loader_B, criterion)
print(f"\nInitial model (trained on A) accuracy on B (new classes): {val_acc_B_init:.3f}")

# -----------------------------
# 9. Stateless Retraining (A + B from scratch)
# -----------------------------
model_stateless = init_model()
optimizer_stateless = optim.Adam(model_stateless.parameters(), lr=1e-3)

epochs_stateless = 5

print("\n=== Stateless Retraining on A + B ===")
for epoch in range(epochs_stateless):
    train_loss, train_acc = train_one_epoch(model_stateless, train_loader_combined, optimizer_stateless, criterion)
    val_loss, val_acc = evaluate(model_stateless, val_loader_combined, criterion)
    print(f"[Stateless] Epoch {epoch+1}/{epochs_stateless} | "
          f"Train Acc: {train_acc:.3f} | Val Acc (A+B): {val_acc:.3f}")

_, acc_A_stateless = evaluate(model_stateless, val_loader_A, criterion)
_, acc_B_stateless = evaluate(model_stateless, val_loader_B, criterion)
print(f"Stateless model - Val Acc A: {acc_A_stateless:.3f}, Val Acc B: {acc_B_stateless:.3f}")

# -----------------------------
# 10. Stateful Training (continue from A, train on B)
# -----------------------------
model_stateful = init_model()
model_stateful.load_state_dict(model_A.state_dict())

optimizer_stateful = optim.Adam(model_stateful.parameters(), lr=1e-3)
epochs_stateful = 5

print("\n=== Stateful Training: Continue from A on B ===")
for epoch in range(epochs_stateful):
    train_loss, train_acc = train_one_epoch(model_stateful, train_loader_B, optimizer_stateful, criterion)
    val_loss_B, val_acc_B = evaluate(model_stateful, val_loader_B, criterion)
    val_loss_A, val_acc_A = evaluate(model_stateful, val_loader_A, criterion)
    print(f"[Stateful] Epoch {epoch+1}/{epochs_stateful} | "
          f"Val Acc A: {val_acc_A:.3f} | Val Acc B: {val_acc_B:.3f}")

# -----------------------------
# 11. A/B Testing Simulation
# -----------------------------
print("\n=== A/B Testing Simulation (Test Set) ===")

test_loader = DataLoader(test_full, batch_size=1, shuffle=True)

def get_predictions(model, loader, max_samples=2000):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= max_samples:
                break
            x, y = x.to(device), y.to(device)
            out = model(x)
            _, p = out.max(1)
            preds.append(p.item())
            labels.append(y.item())
    return np.array(preds), np.array(labels)

preds_stateless, labels_stateless = get_predictions(model_stateless, test_loader)
preds_stateful, labels_stateful   = get_predictions(model_stateful, test_loader)

n = len(labels_stateless)
group_mask = np.random.rand(n) < 0.5  # True → Group A (stateless), False → Group B (stateful)

groupA_preds = preds_stateless[group_mask]
groupA_labels = labels_stateless[group_mask]

groupB_preds = preds_stateful[~group_mask]
groupB_labels = labels_stateful[~group_mask]

acc_A = (groupA_preds == groupA_labels).mean()
acc_B = (groupB_preds == groupB_labels).mean()

print(f"A/B Test - Group A (stateless) Acc: {acc_A:.3f}")
print(f"A/B Test - Group B (stateful)  Acc: {acc_B:.3f}")

# -----------------------------
# 12. Bandit Algorithm (ε-greedy) for Model Selection
# -----------------------------
print("\n=== Bandit Algorithm (ε-greedy) for Model Selection ===")

epsilon = 0.1
n_arms = 2
Q = np.zeros(n_arms)   # estimated value of each arm
N = np.zeros(n_arms)   # number of times each arm was chosen

def choose_arm(Q, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(0, n_arms)  # explore
    else:
        return np.argmax(Q)                  # exploit

def update_Q(Q, N, arm, reward):
    N[arm] += 1
    Q[arm] += (reward - Q[arm]) / N[arm]
    return Q, N

test_loader_bandit = DataLoader(test_full, batch_size=1, shuffle=True)

def predict_with_model(model, x):
    model.eval()
    with torch.no_grad():
        x = x.to(device)
        out = model(x)
        _, p = out.max(1)
    return p.item()

rewards = []
steps = 2000  # number of simulated requests

for i, (x, y) in enumerate(test_loader_bandit):
    if i >= steps:
        break
    y = y.item()

    arm = choose_arm(Q, epsilon)

    if arm == 0:
        pred = predict_with_model(model_stateless, x)
    else:
        pred = predict_with_model(model_stateful, x)

    reward = 1 if pred == y else 0
    rewards.append(reward)

    Q, N = update_Q(Q, N, arm, reward)

avg_reward = np.mean(rewards)
print("Final Q estimates (value per model):", Q)
print("Arm counts (how often each model used):", N)
print("Average reward (accuracy proxy):", avg_reward)

print("\n=== Done ===")

Device: cpu


100%|██████████| 170M/170M [00:14<00:00, 11.5MB/s]


Train size: 50000
Test size: 10000
Subset A size: 20000
Subset B size: 20000
A train/val: 16000 4000
B train/val: 16000 4000

=== Initial Training on Subset A (classes 0–4) ===
[Initial A] Epoch 1/5 | Train Acc: 0.552 | Val Acc (A): 0.624
[Initial A] Epoch 2/5 | Train Acc: 0.673 | Val Acc (A): 0.691
[Initial A] Epoch 3/5 | Train Acc: 0.708 | Val Acc (A): 0.707
[Initial A] Epoch 4/5 | Train Acc: 0.740 | Val Acc (A): 0.739
[Initial A] Epoch 5/5 | Train Acc: 0.769 | Val Acc (A): 0.740

Initial model (trained on A) accuracy on B (new classes): 0.000

=== Stateless Retraining on A + B ===
[Stateless] Epoch 1/5 | Train Acc: 0.464 | Val Acc (A+B): 0.548
[Stateless] Epoch 2/5 | Train Acc: 0.603 | Val Acc (A+B): 0.615
[Stateless] Epoch 3/5 | Train Acc: 0.666 | Val Acc (A+B): 0.645
[Stateless] Epoch 4/5 | Train Acc: 0.711 | Val Acc (A+B): 0.669
[Stateless] Epoch 5/5 | Train Acc: 0.742 | Val Acc (A+B): 0.672
Stateless model - Val Acc A: 0.632, Val Acc B: 0.712

=== Stateful Training: Continue fro